In [1]:
from typing import Dict, List, Tuple
import re
import random as rm
import pandas as pd
import numpy as np
import os
from collections import defaultdict

from config import DATA_PATHS, VALID_CLASSES
from data_utils import (
    load_labels,
    clean_all_patch_files,
    get_all_patch_files,
    group_patches_by_slice,
    build_slice_to_class_map,
    build_case_dict,
)

In [2]:
labels = load_labels(DATA_PATHS['labels_csv'])
filtered_labels = labels[labels['Class'].isin(VALID_CLASSES)].set_index('Case')
filtered_labels['Class'] = filtered_labels['Class'].replace({1: 0, 3: 1, 4: 1})

patch_dict = lambda: defaultdict(patch_dict)
patch_data = patch_dict()
for filename in get_all_patch_files(clean_all_patch_files(), DATA_PATHS['patches_dir']):
    match = re.match(r'case_([^0]\d*)_(\w+_\d+)_(h&e|melan|sox10)_patch(\d+).png', filename)
    if int(match.group(1)) in filtered_labels.index:
        class_id = int(filtered_labels.loc[int(match.group(1)), 'Class'])
        case_id = match.group(1)
        stain_id = match.group(3)
        slice_id = match.group(2)
        patch_id = match.group(4)
        patch_data[class_id][case_id][stain_id][slice_id][patch_id] = filename

Found and excluding 1 files not following naming convention:
  case_056_match_1_h&e.png
Found and excluding 64 potentially duplicate files:
  case_43_match_1_h&e_patch105 2.png
  case_43_match_1_sox10_patch70 2.png
  case_43_match_1_h&e_patch8 2.png
  case_43_match_1_melan_patch47 2.png
  case_43_match_1_melan_patch32 2.png
  ... and 59 more
25878 out of 25943 non-standard file names were successfully coerced to standard.


In [3]:
benign_many_slices = pd.DataFrame()
for case_id, stain_id in patch_data[0].items():
    num_slices = {}
    for stain_id, slice_id in stain_id.items(): 
        num_slices[(case_id, stain_id)] = len(slice_id)
    if max(num_slices.values()) > 5: 
        for (case, stain), slices in num_slices.items():
            benign_many_slices.loc[case, stain] = slices

benign_many_slices.sort_index(inplace=True)
benign_many_slices

,h&e,melan,sox10
22,13.0,3.0,3.0
24,10.0,4.0,2.0
25,6.0,2.0,2.0
26,17.0,8.0,4.0
27,6.0,4.0,2.0
82,6.0,1.0,1.0


In [5]:
for case in benign_many_slices.index:
    stains_count = {}
    for stain_id, slice_id in patch_data[0][case].items():
        stains_count[stain_id] = len(slice_id)
    cases_target = int(np.ceil(max(stains_count.values())/5))

    new_case_split = {}
    for stain_id, slice_id in patch_data[0][case].items():
        new_case_split[stain_id] = \
            np.array_split(list(slice_id.keys()), cases_target)

    for i in range(cases_target):
        pseudocase_id = (i+1)*1000+int(case)
        for stain_id, slice_id in patch_data[0][case].items():
            patch_data[0][pseudocase_id][stain_id] = \
                {
                    slice: patch for slice, patch in slice_id.items()
                    if slice in new_case_split[stain_id][i]
                } 
            for slice_id, patch_id in patch_data[0][pseudocase_id][stain_id].items():
                for path in patch_id.values():
                    new_path = re.sub(r'case_\d+_', f'case_{pseudocase_id}_', path)
                    os.replace(
                        os.path.join(DATA_PATHS['patches_dir'], path), 
                        os.path.join(DATA_PATHS['patches_dir'], new_path)
                    )
        with open(DATA_PATHS['labels_csv'], 'a') as f:
            f.write(f"\n{pseudocase_id},1")